# 15. Instance Segmentation - yolov8s_seg

| 항목 | 내용 |
|---|---|
| 대상 모델 | `yolov8s_seg.dxnn` |
| 카테고리 | **Instance Segmentation** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. Public Output 두 개가 서로 다른 Task 에서 나온다는 사실을 Task 구성으로 설명할 수 있다.
2. 계수 32개와 밑그림 32장을 곱해 마스크를 만드는 계산을 shape 변화와 함께 설명할 수 있다.
3. 마스크를 원본 크기로 되돌리는 네 단계를 순서대로 제시할 수 있다.
4. 개체별 마스크와 부류별 마스크의 차이를 같은 장면의 결과로 비교해 설명할 수 있다.

## 1. Instance Segmentation (개체 분할)

**영상에서 대상을 하나하나 찾아 각각의 윤곽을 픽셀 단위로 답하는 작업**

검출이 사각형까지, 의미 분할이 부류별 영역까지 답하는 데 비해 이 작업은 **개체마다 따로** 윤곽을 답한다. 같은
부류의 대상이 여럿 붙어 있어도 각각이 별개의 결과가 되므로 개수를 셀 수 있다.

**출력의 성격**

개체 수만큼의 목록이며 항목마다 사각형·부류·점수·마스크가 함께 붙는다. 마스크는 사각형 안쪽만 유효하도록
잘라내므로 사각형이 틀리면 윤곽도 잘린다.

**활용 사례**

- 겹친 물체의 개수·면적 측정
- 배경 분리·합성 (개체 단위)
- 로봇 집기 위치 판단

**포함되지 않는 작업**

- 부류별 영역만 구분 (Semantic Segmentation)
- 사각형까지만 답하기 (Object Detection)
- 같은 개체를 다른 화면에서 다시 찾기 (Person Re-ID)

## 2. 전체 파이프라인

```text
sample_street.jpg  768×576 BGR
  → 전처리         BGR→RGB · 비율 유지 축소 640×480 · 위아래 여백 80 → 640×640
  → run()          입력 Tensor 하나를 전달
  → npu_0          output1  [1, 32, 160, 160]  float32  (Public Output — 마스크 밑그림)
  → cpu_0          output0  [1, 116, 8400]     float32  (Public Output — 후보 목록)
  → 후처리         모양으로 두 출력 구분 → transpose → 점수·NMS →
                   계수 × 밑그림 → sigmoid → 확대 → 사각형 밖 잘라내기 → 원본 크기
  → Result         InstanceSegResult (box · confidence · class_id · mask)
  → 시각화         개체마다 다른 색 마스크를 40% 로 덧씌우고 사각형·라벨을 그림
```

![15. Instance Segmentation 전체 파이프라인](resources/15_Instance_Segmentation_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `images` |
| Shape / Dtype | `[1, 640, 640, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | **Letterbox — 비율 유지 후 여백 채우기** |
| Normalization | 없음 |

12번·13번과 같은 입력이다. 좌표와 마스크를 모두 되돌려야 하므로 `gain`·`pad_x`·`pad_y` 를 Context 에 담아 둔다.
이번 실행의 `gain` 은 `0.833333`, 위아래 여백은 80 씩이다.

### 3.2 출력

**Public Output 이 두 개이고 서로 다른 Task 에서 나온다.** 마스크 밑그림은 `npu_0` 이 직접 내보내고, 후보 목록은
`cpu_0` 이 여러 중간 값을 합쳐 만든다.

| 생성 Task | Tensor | shape / dtype | 다음 사용자 | Public |
|---|---|---|---|---|
| `npu_0` | `output1` | `[1, 32, 160, 160]` / `float32` | Host 후처리 | **O — NPU Task 출력** |
| `npu_0` | `cv2·cv3·cv4` 9개 | `[1, 64·80·32, 격자]` / `float32` | `cpu_0` | 아님 — 내부 전달 |
| `cpu_0` | `output0` | `[1, 116, 8400]` / `float32` | Host 후처리 | **O — CPU Task 출력** |

하나의 `.dxnn` 에서 **NPU 가 직접 내는 값과 CPU Task 가 만든 값이 함께** 나오는 구성이다. 그래서 후처리는 받은
목록의 순서를 믿지 않고 **모양으로 어느 것이 무엇인지 판정한다** — 4차원이고 채널이 32면 밑그림, 3차원이고 한 축이
116 이면 후보 목록이다.

**후보 한 줄 116개 값의 자리**

| index | 개수 | 내용 |
|---:|---:|---|
| 0~3 | 4 | 중심 좌표와 크기 `cx, cy, w, h` (입력 640 기준) |
| 4~83 | 80 | 부류마다의 점수 |
| 84~115 | 32 | **마스크 계수** |

`116 = 4 + 80 + 32` 이며 objectness 자리가 없다. 14번과 같이 YOLOv8 계열이라 부류 점수의 최댓값이 곧 최종 점수다.
후보 수 8,400 도 세 격자의 합(`80×80 + 40×40 + 20×20`)이며 자리당 후보가 하나다.

**마스크는 밑그림 32장을 계수로 섞어 만든다.** 개체마다 마스크 한 장을 통째로 내보내면 후보 8,400개 × 160×160 이
되어 감당할 수 없다. 대신 **모든 개체가 공유하는 밑그림 32장**을 한 번만 내보내고, 개체마다 그것을 어떻게 섞을지
정하는 **계수 32개**만 따로 갖는다.

```text
mask = sigmoid( 계수[32] · 밑그림[32, 160×160] )     →  [160, 160]
```

계수 하나가 밑그림 한 장의 가중치이며, 곱셈 결과를 sigmoid 로 0~1 로 만들어 0.5 를 넘는 화소를 개체 영역으로
본다. 이번 실행에서 계수의 범위는 `-2.64`~`1.70`, 섞은 뒤 확률의 범위는 `0.0000`~`0.9998` 이다.

**밑그림은 입력의 1/4 크기다.** 160×160 이며 `640 / 160 = 4` 다. 작게 만들어 계산을 줄이고 나중에 확대하는 구조라
**윤곽이 4화소 단위로 뭉뚱그려진다** — 개체 분할의 경계가 의미 분할보다 거친 이유다.

**80개 부류의 번호는 12·13번과 같은 COCO 목록**을 쓴다.

| index | 이름 | 고른 이유 |
|---:|---|---|
| 0 | person | 목록의 첫 항목 |
| 1 | bicycle | 이번 실행 검출 |
| 2 | car | 12번에서 검출된 부류 |
| 7 | truck | 이번 실행 검출 |
| 9 | traffic light | 도로 장면에 흔한 부류 |
| 16 | dog | 이번 실행 1위 |
| 58 | potted plant | 이번 실행 최저 점수 검출 |
| 77 | teddy bear | 끝에서 세 번째 |
| 78 | hair drier | 끝에서 두 번째 |
| 79 | toothbrush | 목록의 마지막 항목 |

전체 목록: [ultralytics coco.yaml](https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/datasets/coco.yaml)

#### 출력을 사람이 읽는 값으로 바꾸기

마스크는 밑그림 크기(160×160)에서 만들어지므로 원본 사진에 얹으려면 네 단계를 거친다. **순서를 바꾸면 마스크가
어긋난다.**

| 순서 | 하는 일 | 이유 |
|---:|---|---|
| 1 | 입력 크기(640×640)로 확대 | 사각형 좌표와 같은 좌표계로 맞춘다 |
| 2 | 사각형 밖을 0 으로 | 밑그림 조합은 사각형 밖에도 값이 남는다 |
| 3 | 여백을 잘라 냄 | letterbox 가 넣은 띠를 뺀다 |
| 4 | 원본 크기로 확대 | 사진 위에 그릴 수 있게 한다 |

2단계를 빼면 다른 개체의 영역까지 칠해지고, 3단계를 빼면 마스크 전체가 위로 밀린다.

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. `npu_0` 에 `(model output)`
표시가 붙어 `output1` 이 직접 나오고, `cpu_0` 이 `output0` 을 내는지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "yolov8s_seg.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

### 3.3 Config.json

| key | 현재 값 | 기본값 | 의미 |
|---|---:|---:|---|
| `obj_threshold` | 0.25 | 0.25 | **이 배포본에서 쓰이지 않음** |
| `score_threshold` | 0.25 | 0.3 | 최종 점수가 이 값에 못 미치는 후보를 버림 |
| `nms_threshold` | 0.45 | 0.45 | 겹침이 이 비율을 넘으면 낮은 점수를 버림 |

**`obj_threshold` 는 YOLOv5 계열용이다.** 같은 후처리 코드가 두 계열을 모두 받는데, objectness 자리가 있는 v5 에서만
이 값으로 먼저 걸러낸다. v8 은 그 자리가 없어 값이 읽히더라도 쓰이지 않는다.

`score_threshold` 는 코드 기본값 `0.3` 을 `0.25` 로 낮춰 둔다. 낮추면 흐릿한 개체도 남고 잘못된 마스크도 늘어난다.

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정하고 설정 파일과 COCO 80개 이름, 개체별 색 20개를 읽는다. 이후 셀은
여기서 만든 이름을 그대로 쓴다.

In [ ]:
import json
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

IMAGE_PATH = DX_APP / "sample" / "img/sample_street.jpg"
CONFIG_PATH = (DX_APP / "src" / "python_example" / "instance_segmentation" / "yolov8s_seg"
               / "config.json")
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}

# dx-all-suite 는 읽기 전용 원본이다. 바이트코드 캐시를 남기지 않도록 먼저 꺼 둔다.
sys.dont_write_bytecode = True
sys.path.insert(0, str(DX_APP / "src" / "python_example"))
from common.utility import get_labels  # noqa: E402

COCO_LABELS = get_labels("coco80")
NUM_MASKS = 32
INSTANCE_COLORS = [
    (255, 56, 56), (255, 157, 151), (255, 112, 31), (255, 178, 29),
    (207, 210, 49), (72, 249, 10), (146, 204, 23), (61, 219, 134),
    (26, 147, 52), (0, 212, 187), (44, 153, 168), (0, 194, 255),
    (52, 69, 147), (100, 115, 255), (0, 24, 236), (132, 56, 255),
    (82, 0, 133), (203, 56, 255), (255, 149, 200), (255, 55, 199),
]

print("config", CONFIG)
print("classes", len(COCO_LABELS), "/ masks", NUM_MASKS, "/ colors", len(INSTANCE_COLORS))

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 좌표와 마스크를 모두 되돌리는 데
  `scale`(gain)·`pad_x`·`pad_y` 를 쓴다
- `InstanceSegResult` — 후처리가 만들어 시각화로 넘기는 결과. 사각형·점수·부류·마스크를 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale: float = 1.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class InstanceSegResult:
    """개체 하나의 사각형·점수·부류·마스크. 원본 대응: common/base/i_processor.py"""

    box: list = field(default_factory=list)
    confidence: float = 0.0
    class_id: int = 0
    mask: np.ndarray = field(default=None)


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. 출력이 두 개이고 shape 가 서로 다른 것을
확인한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output (런타임 순서):")
for index, info in enumerate(output_info):
    print(f"  [{index}] {info['name']:10s} {info['shape']}")

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

비율을 유지해 640×480 으로 줄이고 위아래에 80씩 회색 여백을 넣는다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 640×640, 그중 실제 사진은 640×480
- Context — `scale`(gain) 과 `pad_x`·`pad_y`

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다. 12·13번과 같은 사진이므로 검출 결과를 비교할 수 있다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}", figsize=(10, 7))

#### 전처리 함수 정의

- `seg_preprocessor_create` — 모델 입력 크기와 여백 색을 state 에 담음
- `seg_letterbox` — 비율 유지 축소와 여백 채우기
- `seg_preprocessor_process` — letterbox 를 적용하고 복원용 값을 Context 에 담음

In [ ]:
def seg_preprocessor_create(input_width: int, input_height: int,
                            pad_color: tuple = (114, 114, 114)) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height, "pad_color": pad_color}


def seg_letterbox(state: dict, image: np.ndarray) -> tuple:
    """비율을 유지해 줄이고 남는 자리를 채운다. 원본 대응: LetterboxPreprocessor._letterbox()"""
    height, width = image.shape[:2]
    gain = min(state["input_height"] / height, state["input_width"] / width)
    new_size = int(round(width * gain)), int(round(height * gain))

    delta_w = (state["input_width"] - new_size[0]) / 2
    delta_h = (state["input_height"] - new_size[1]) / 2
    if (width, height) != new_size:
        image = cv2.resize(image, new_size, interpolation=cv2.INTER_LINEAR)

    top, bottom = int(round(delta_h - 0.1)), int(round(delta_h + 0.1))
    left, right = int(round(delta_w - 0.1)), int(round(delta_w + 0.1))
    padded = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT,
                                value=state["pad_color"])
    return padded, gain, (top, left)


def seg_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    padded, gain, pad = seg_letterbox(state, rgb)
    ctx.scale = gain
    ctx.pad_y, ctx.pad_x = pad
    return padded, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다. 여백을 뺀 실제 사진 영역의 크기를 함께 출력해 두면 마스크를 되돌릴 때 쓰는 값이
무엇인지 분명해진다.

In [ ]:
preprocessor_state = seg_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = seg_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"gain={ctx.scale:.6f} pad_x={ctx.pad_x} pad_y={ctx.pad_y}")
print(f"여백 뺀 영역   {int(round(ctx.original_width * ctx.scale))}x"
      f"{int(round(ctx.original_height * ctx.scale))}")

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 넘기고 Public Output 두 개를 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복 비용이
포함돼 있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(f"  [{index}] {output_info[index]['name']:10s} {str(array.shape):20s} "
          f"min={array.min():8.4f} max={array.max():8.4f} {array.nbytes:,} bytes")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- 두 출력의 차원 수 — 4차원이면 밑그림, 3차원이면 후보 목록
- 후보 목록의 항목 수 — `116 = 4 + 80 + 32`
- 밑그림의 크기 — 160×160 이 입력 640 의 몇 분의 1인지
- 마스크 계수의 범위 — 0~1 이 아니라 음수도 있다

In [ ]:
detection_raw = proto_raw = None
for output in outputs:
    array = np.asarray(output)
    if array.ndim == 4 and (array.shape[1] == NUM_MASKS or array.shape[-1] == NUM_MASKS):
        proto_raw = array
    elif array.ndim == 3 and (4 + len(COCO_LABELS) + NUM_MASKS) in (array.shape[1], array.shape[2]):
        detection_raw = array

print(f"모양으로 판정: 후보 목록 {detection_raw.shape} / 밑그림 {proto_raw.shape}")

squeezed = np.squeeze(detection_raw)
candidates = np.transpose(squeezed) if squeezed.shape[0] < squeezed.shape[1] else squeezed
print(f"transpose 뒤 {candidates.shape}  항목 {candidates.shape[1]} = 4 + {len(COCO_LABELS)} + {NUM_MASKS}")

class_scores = candidates[:, 4:4 + len(COCO_LABELS)]
mask_coefficients = candidates[:, 4 + len(COCO_LABELS):]
confidences = class_scores.max(axis=1)

proto = np.squeeze(proto_raw)
print(f"밑그림 {proto.shape} → 160 은 입력 {INPUT_WIDTH} 의 1/{INPUT_WIDTH // proto.shape[-1]}")
print(f"부류 점수 범위    {class_scores.min():.6f} ~ {class_scores.max():.6f}")
print(f"마스크 계수 범위  {mask_coefficients.min():.4f} ~ {mask_coefficients.max():.4f}")
print(f"점수 {CONFIG.get('score_threshold', 0.3)} 초과 후보 "
      f"{int((confidences >= CONFIG.get('score_threshold', 0.3)).sum())}개 / {candidates.shape[0]:,}개")

### 4.4 후처리

두 출력을 모양으로 구분하고, 후보를 점수·겹침으로 걸러 낸 뒤, 남은 개체마다 마스크를 만들어 원본 좌표계로
되돌린다.

**마스크를 만드는 계산이 이 후처리의 핵심이다.** 남은 개체의 계수만 골라 밑그림과 한 번의 행렬곱으로 처리한다.
개체가 4개면 `[4, 32] × [32, 25600] = [4, 25600]` 이고 이것을 `[4, 160, 160]` 으로 되돌린다. 개체마다 따로
계산하지 않으므로 개체가 늘어도 비용이 크게 늘지 않는다.

**사각형 밖을 0 으로 만드는 단계를 빼면 안 된다.** 밑그림 조합은 사진 전체에 값이 퍼져 있어, 그 개체의 사각형
밖에도 0.5 를 넘는 화소가 남는다. 사각형으로 잘라 내야 개체별 마스크가 된다.

#### 후처리 함수 정의 — 후보 걸러내기

- `seg_postprocessor_create` — 임계값·부류 수·밑그림 수를 state 에 담음
- `seg_align_outputs` — 모양으로 후보 목록과 밑그림을 구분
- `seg_select_candidates` — transpose·점수 계산·NMS 로 남길 개체를 고름

In [ ]:
def seg_postprocessor_create(config: dict) -> dict:
    """Config 와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {
        "score_threshold": config.get("score_threshold", 0.3),
        "nms_threshold": config.get("nms_threshold", 0.45),
        "num_classes": config.get("num_classes", 80),
        "num_masks": config.get("num_masks", 32),
    }


def seg_align_outputs(state: dict, outputs: list) -> tuple:
    """모양으로 후보 목록과 밑그림을 구분한다. 원본 대응: InstanceSegPostprocessor._align_outputs()"""
    expected = 4 + state["num_classes"] + state["num_masks"]
    detection = proto = None
    for output in outputs:
        array = np.asarray(output)
        if array.ndim == 4 and (array.shape[1] == state["num_masks"]
                                or array.shape[-1] == state["num_masks"]):
            proto = array
        elif array.ndim == 3 and expected in (array.shape[1], array.shape[2]):
            detection = array
    return detection, proto


def seg_select_candidates(state: dict, detection_raw: np.ndarray) -> tuple:
    """transpose·점수 계산·NMS 로 남길 개체를 고른다. 원본 대응: InstanceSegPostprocessor.process()"""
    squeezed = np.squeeze(detection_raw)
    output = np.transpose(squeezed) if squeezed.shape[0] < squeezed.shape[1] else squeezed

    class_scores = output[:, 4:4 + state["num_classes"]]
    confidences = class_scores.max(axis=1)
    class_ids = class_scores.argmax(axis=1)
    coefficients = output[:, 4 + state["num_classes"]:]

    corners = np.column_stack([
        output[:, 0] - output[:, 2] * 0.5, output[:, 1] - output[:, 3] * 0.5,
        output[:, 0] + output[:, 2] * 0.5, output[:, 1] + output[:, 3] * 0.5,
    ])
    boxes_xywh = np.column_stack([corners[:, 0], corners[:, 1],
                                  corners[:, 2] - corners[:, 0], corners[:, 3] - corners[:, 1]])
    indices = cv2.dnn.NMSBoxes(boxes_xywh.tolist(), confidences.tolist(),
                               state["score_threshold"], state["nms_threshold"])
    keep = np.array(indices).reshape(-1) if len(indices) else np.empty(0, dtype=int)
    return keep, corners, confidences, class_ids, coefficients

#### 후처리 함수 정의 — 마스크 만들기

- `seg_build_masks` — 계수와 밑그림을 곱해 마스크를 만들고 입력 크기로 확대한 뒤 사각형 밖을 0 으로
- `seg_mask_to_original` — 여백을 잘라 내고 원본 크기로 확대
- `seg_postprocessor_process` — 위 함수들을 순서대로 불러 Result 생성

In [ ]:
def seg_build_masks(state: dict, coefficients: np.ndarray, proto_raw: np.ndarray,
                    corners: np.ndarray) -> np.ndarray:
    """계수 × 밑그림으로 마스크를 만들고 입력 크기로 확대한 뒤 사각형 밖을 0 으로 만든다.

    원본 대응: InstanceSegPostprocessor._generate_scaled_masks()
    """
    proto = np.squeeze(proto_raw)
    if proto.ndim == 3 and proto.shape[-1] == state["num_masks"] and proto.shape[0] != state["num_masks"]:
        proto = np.transpose(proto, (2, 0, 1))
    channels, mask_h, mask_w = proto.shape

    combined = 1.0 / (1.0 + np.exp(-(coefficients @ proto.reshape(channels, -1))))
    masks = combined.reshape(-1, mask_h, mask_w)

    scaled = np.zeros((len(masks), state["input_height"], state["input_width"]), dtype=np.float32)
    for index, mask in enumerate(masks):
        scaled[index] = cv2.resize(mask, (state["input_width"], state["input_height"]),
                                   interpolation=cv2.INTER_LINEAR)

    for index, box in enumerate(corners):
        x1, y1, x2, y2 = box.astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(state["input_width"], x2), min(state["input_height"], y2)
        scaled[index, :y1, :] = 0
        scaled[index, y2:, :] = 0
        scaled[index, :, :x1] = 0
        scaled[index, :, x2:] = 0
    return scaled


def seg_mask_to_original(mask: np.ndarray, ctx: PreprocessContext) -> np.ndarray:
    """여백을 잘라 내고 원본 크기로 확대한다. 원본 대응: InstanceSegPostprocessor._crop_mask_to_original()"""
    gain = max(ctx.scale, 1e-6)
    unpad_h = int(round(ctx.original_height * gain))
    unpad_w = int(round(ctx.original_width * gain))
    cropped = mask[ctx.pad_y:ctx.pad_y + unpad_h, ctx.pad_x:ctx.pad_x + unpad_w]
    if cropped.size == 0:
        return np.zeros((ctx.original_height, ctx.original_width), dtype=np.float32)
    return cv2.resize(cropped, (ctx.original_width, ctx.original_height),
                      interpolation=cv2.INTER_LINEAR)


def seg_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    detection_raw, proto_raw = seg_align_outputs(state, outputs)
    if detection_raw is None or proto_raw is None:
        return []

    keep, corners, confidences, class_ids, coefficients = seg_select_candidates(state, detection_raw)
    if len(keep) == 0:
        return []

    state = {**state, "input_width": ctx.input_width, "input_height": ctx.input_height}
    masks = seg_build_masks(state, coefficients[keep], proto_raw, corners[keep])

    gain = max(ctx.scale, 1e-6)
    results = []
    for order, index in enumerate(keep):
        box = corners[index]
        original_mask = seg_mask_to_original(masks[order], ctx)
        results.append(InstanceSegResult(
            box=[float(np.clip((box[0] - ctx.pad_x) / gain, 0, ctx.original_width - 1)),
                 float(np.clip((box[1] - ctx.pad_y) / gain, 0, ctx.original_height - 1)),
                 float(np.clip((box[2] - ctx.pad_x) / gain, 0, ctx.original_width - 1)),
                 float(np.clip((box[3] - ctx.pad_y) / gain, 0, ctx.original_height - 1))],
            confidence=float(confidences[index]), class_id=int(class_ids[index]),
            mask=(original_mask > 0.5).astype(np.uint8)))
    return results

#### 후처리 실행

확인할 것:

- 단계별 개수 — 8,400 → 점수·NMS 통과
- 각 개체의 부류·점수·사각형 — 원본 크기 안에 들어오는지
- 마스크 shape — 원본 크기와 같은지
- 마스크 화소 수 — 사각형 면적보다 작은지 (윤곽이 사각형을 다 채우지 않음)

In [ ]:
postprocessor_state = seg_postprocessor_create(CONFIG)
results = seg_postprocessor_process(postprocessor_state, outputs, ctx)

print(f"후보 {candidates.shape[0]:,}개 → 점수 {postprocessor_state['score_threshold']} · "
      f"NMS {postprocessor_state['nms_threshold']} 통과 {len(results)}개")

for order, result in enumerate(sorted(results, key=lambda r: -r.confidence), start=1):
    x1, y1, x2, y2 = result.box
    box_area = max(1, int((x2 - x1) * (y2 - y1)))
    mask_area = int(result.mask.sum())
    print(f"  #{order} {COCO_LABELS[result.class_id]:13s} conf={result.confidence:.4f} "
          f"box=({x1:6.1f}, {y1:6.1f}, {x2:6.1f}, {y2:6.1f}) "
          f"mask={mask_area:7,}화소 (사각형의 {mask_area / box_area * 100:4.1f}%)")

print(f"\nmask shape={results[0].mask.shape} dtype={results[0].mask.dtype} "
      f"(원본 {ctx.original_width}x{ctx.original_height} 와 같다)")

### 4.5 시각화

#### Visualizer 코드

결과 객체만 받아 개체마다 다른 색으로 마스크를 칠하고 40% 로 섞은 뒤 사각형과 라벨을 그린다. **부류가 아니라
개체 순서로 색을 고르므로** 같은 부류의 두 개체가 다른 색이 된다. 그림에 넣는 글자는 영문만 쓴다 — OpenCV 기본
폰트에 한글 자모가 없어 한글을 넣으면 글자가 깨진다.

In [ ]:
def seg_visualizer_visualize(image: np.ndarray, results: list, labels: list,
                             alpha: float = 0.4) -> np.ndarray:
    """Result 를 원본 이미지 위에 그린다. 원본 대응: Runner.visualize()"""
    output = image.copy()
    overlay = image.copy()

    for order, result in enumerate(results):
        color = INSTANCE_COLORS[order % len(INSTANCE_COLORS)]
        x1, y1, x2, y2 = [int(value) for value in result.box]

        if result.mask is not None and result.mask.shape[:2] == image.shape[:2]:
            overlay[result.mask > 0] = color

        cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)
        name = labels[result.class_id] if result.class_id < len(labels) else str(result.class_id)
        text = f"{name} {result.confidence:.2f}"
        (text_w, text_h), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(output, (x1, y1 - text_h - 6), (x1 + text_w, y1), color, -1)
        cv2.putText(output, text, (x1, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    return cv2.addWeighted(overlay, alpha, output, 1 - alpha, 0)

#### 결과 시각화

개체별 마스크를 덧씌운 결과를 inline 으로 표시한다. 같은 부류가 여럿이면 색이 서로 다른지 확인한다.

In [ ]:
output_image = seg_visualizer_visualize(image, results, COCO_LABELS)
notebook_show_bgr(output_image, f"{len(results)} instances with masks", figsize=(11, 8))

#### 개체별 마스크를 따로 보기

겹쳐 그린 그림에서는 어느 화소가 어느 개체인지 가려진다. 개체마다 마스크만 흰색으로 그려 나란히 놓으면 **각각이
독립된 한 장**이라는 점과 사각형으로 잘린 경계가 드러난다.

In [ ]:
panels = []
for order, result in enumerate(sorted(results, key=lambda r: -r.confidence)):
    panel = cv2.cvtColor(result.mask * 255, cv2.COLOR_GRAY2BGR)
    cv2.putText(panel, f"{COCO_LABELS[result.class_id]} {result.confidence:.2f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2, cv2.LINE_AA)
    panels.append(cv2.resize(panel, (300, int(300 * image.shape[0] / image.shape[1]))))

print(f"개체 {len(panels)}개의 마스크를 따로 그린다. 흰색이 그 개체의 영역이다.")
notebook_show_bgr(cv2.hconcat(panels), "Per-instance masks", figsize=(14, 4))